In [84]:
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from rapidfuzz import process, fuzz

In [85]:
## DON'T CHANGE THIS LINE
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # because some act names are in Hindi and English

In [86]:

class StatuteExtractor:
    def __init__(self, act_names=None, common_abbreviations=None):
        # Compile possible act names (for strict matching), or use a generic fallback if none provided
        possible_act_names = []
        if act_names:
            for act in act_names:
                cleaned_act = re.sub(r'^\bThe\b\s+', '', act, flags=re.IGNORECASE)
                cleaned_act = re.sub(r',?\s+\d{4}$', '', cleaned_act)
                possible_act_names.append(cleaned_act)
            possible_act_names += act_names
        if common_abbreviations:
            possible_act_names += common_abbreviations
        # Remove empties and dedupe
        possible_act_names = [x for x in set(possible_act_names) if x]
        if possible_act_names:
            law_list_regex = r'(?:' + '|'.join(sorted(map(re.escape, possible_act_names), key=len, reverse=True)) + r')'
        else:
            # fallback to generic match for "Act", "Code", "Law", "Rules", "Regulations" etc.
            law_list_regex = r'[A-Za-z].*?(?:Act|Code|Law|Regulation|Rules|Ordinance|Bill|Amendment|Notification|Order|Guidelines|IPC|CrPC|CPC)'
        # Pattern
        self.pattern = re.compile(
            r'''
            \b
            (?:(?:[Ss]ection|[Aa]rticle|[Ss]chedule)s?|[Uu]/[Ss])            # “Section(s)” or “Article(s)” or “u/s”
            \W+                                                  # separator (spaces, hyphens…)
            (?P<sections>                                        # one or more section‑number tokens
                (?:
                    (?:[Ss]ection|[Aa]rticle|[Ss]chedule\s*)?
                    \d+(?:\s*\(([A-Za-z]{1,2}|[0-9]{1,3})\)|[-_]([A-Za-z]{1,2}|[0-9]{1,3}))?
                )
                (?:
                    \s*(?:,\s*|\s+and\s+)
                    (?:[Ss]ection|[Aa]rticle|[Ss]chedule\s*)?
                    \d+(?:\s*\(([A-Za-z]{1,2}|[0-9]{1,3})\)|[-_]([A-Za-z]{1,2}|[0-9]{1,3}))?
                )*
            )
            \s+
            ,?\s*
            (?:of\s+)?                                          # optional “of”
            (?:the\s+)?                                         # optional “the”
            (?P<law_name>
                (?:Code\s+(?:of|on|for|to)(?:\s+[A-Z][a-zA-Z]*)+)
              |
                (?:[A-Z][a-zA-Z]*(?:\s+[A-Z][a-zA-Z]*)*\s+(?:Act|Code|Law|Regulation|Rules|Ordinance|Bill|Amendment|Notification|Order|Guidelines|Adhiniyam))
              |
                ''' + law_list_regex + r'''
              |
                (?:[A-Za-z]{1,2}\.)+         # one or more “XX.” segments
                (?:[A-Za-z]{1,2}\.?)?        # optional final "XX" with optional dot
            )
            \b
            ''',
            flags=re.IGNORECASE | re.VERBOSE
        )

    def normalize_section_name(self, section):
        section_name = section.lower()  # Normalize to lowercase
        section_name = re.sub(r'[^A-Za-z0-9]', '_', str(section_name))
        section_name = re.sub(r'([A-Za-z])([0-9])', r'\1_\2', section_name)
        section_name = re.sub(r'([0-9])([A-Za-z])', r'\1_\2', section_name)
        section_name = re.sub(r'_+', '_', section_name)
        section_name = section_name.strip('_')
        section_name = re.sub(r'(^_?0\b|(?<=_)0\b|^0_|\b0_?$)', '', section_name)
        section_name = re.sub(r'_+', '_', section_name).strip('_')
        return section_name

    def extract(self, text):
        """Returns a list of {'section': ..., 'act': ...} elements."""
        matches = self.pattern.finditer(text)
        results = []
        seen = set()
        for match in matches:
            sections_chunk = match.group('sections')
            law_name = match.group('law_name').strip()
            raw_sections = re.split(r'(?:,|\band\b)', sections_chunk)
            for sect in raw_sections:
                sect = sect.strip()
                if sect:
                    sect = re.sub(r'^[Ss]ection\s*', '', sect)
                    sect = re.sub(r'^[Aa]rticle\s*', '', sect)
                    sect = self.normalize_section_name(sect)
                    name = f"section_{sect}_of_{law_name}"
                    if name in seen:
                        continue
                    seen.add(name)
                    results.append({'section': sect, 'act': law_name})
        return results

In [87]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)     # standardize whitespace
    return text.strip()

def match_statute_name(user_input, act_embeddings, act_names, return_top_k=3, score_threshold=0.73):
    """Return top-k best matches and their similarity scores"""
    norm_query = normalize(user_input)
    query_emb = model.encode([norm_query], convert_to_numpy=True, normalize_embeddings=True)

    scores = np.dot(act_embeddings, query_emb[0])
    ranked_idx = np.argsort(scores)[::-1]  # Descending order

    # Filter by threshold and return top K
    results = []
    for idx in ranked_idx[:return_top_k]:
        score = scores[idx]
        if score < score_threshold:
            continue
        results.append({'act_name': act_names[idx], 'score': float(score)})
    return results

In [ ]:
# Load embeddings
act_embeddings = np.load("act_embeddings.npy")

# Load mapping
with open("act_names.json", "r", encoding="utf-8") as f:
    act_names = json.load(f)

with open("./data/final_cleaned_alt.json", "r", encoding="utf-8") as f:
    kachha_data = json.load(f) ## raw data
    
with open("./data/statutes_data/act_2_id.json", "r", encoding="utf-8") as f:
    act_2_id = json.load(f)

with open("./data/statutes_data/id_2_text.json", "r", encoding="utf-8") as f:
    id_2_text = json.load(f)

with open("./data/statutes_data/ipc_sections.json", "r", encoding="utf-8") as f:
    IPC_dict = json.load(f)

with open("./data/statutes_data/crpc_sections.json", "r", encoding="utf-8") as f:
    CrPC_dict = json.load(f)

common_abbreviations = [
    'IPC', 'CrPC', 'CPC', 'IT Act', 'RTI Act', 'POSCO', 'SC/ST Act', 'PoA', 'FCRA', 'FEMA', 'PMLA', 'NDPS', 'HMA', 'DV Act',
    'DVC', 'NIA', 'RERA', 'DRT', 'CGST Act', 'SEBI Act', 'MVA', 'ESMA', 'POTA', 'PoTA', 'COFEPOSA', 'CoFEPoSA', 'CAT', 'PSA',
    'UAPA', 'NSA', 'MCOCA', 'SARFAESI', 'SARFAESI Act', 'JJ Act', 'SHWW Act', 'POSH Act', 'SHWW', 'POSH', 'PoSH', 'FEOA', 'GI Act',
    'PCA', 'NDPS', 'NDPS Act', 'Dowry Act', 'Dowry Proh. Act', 'IEA', 'UPPRA', 'UPPCA', 'Cow Slaughter Act', 'UPPDA', 'UPPFA', 'Goonda Act', 'Gangsters Act',
]

IPC_list = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '29A', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '52A', '53', '53A', '54', '55', '55A', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '108A', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '120A', '120B', '121', '121A', '122', '123', '124', '124A', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '138A', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '153A', '153AA', '153B', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '165A', '166', '166A', '166B', '167', '168', '169', '170', '171', '171A', '171B', '171C', '171D', '171E', '171F', '171G', '171H', '171I', '172', '173', '174', '174A', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '195A', '196', '197', '198', '199', '200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214', '215', '216', '216A', '216B', '217', '218', '219', '220', '221', '222', '223', '224', '225', '225A', '225B', '226', '227', '228', '228A', '229', '229A', '230', '231', '232', '233', '234', '235', '236', '237', '238', '239', '240', '241', '242', '243', '244', '245', '246', '247', '248', '249', '250', '251', '252', '253', '254', '255', '256', '257', '258', '259', '260', '261', '262', '263', '263A', '264', '265', '266', '267', '268', '269', '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280', '281', '282', '283', '284', '285', '286', '287', '288', '289', '290', '291', '292', '293', '294', '294A', '295', '295A', '296', '297', '298', '299', '300', '301', '302', '303', '304', '304A', '304B', '305', '306', '307', '308', '309', '310', '311', '312', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '326A', '326B', '327', '328', '329', '330', '331', '332', '333', '334', '335', '336', '337', '338', '339', '340', '341', '342', '343', '344', '345', '346', '347', '348', '349', '350', '351', '352', '353', '354', '354A', '354B', '354C', '354D', '355', '356', '357', '358', '359', '360', '361', '362', '363', '363A', '364', '364A', '365', '366', '366A', '366B', '367', '368', '369', '370', '370A', '371', '372', '373', '374', '375', '376', '376A', '376AB', '376B', '376C', '376D', '376DA', '376DB', '376E', '377', '378', '379', '380', '381', '382', '383', '384', '385', '386', '387', '388', '389', '390', '391', '392', '393', '394', '395', '396', '397', '398', '399', '400', '401', '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '412', '413', '414', '415', '416', '417', '418', '419', '420', '421', '422', '423', '424', '425', '426', '427', '428', '429', '430', '431', '432', '433', '434', '435', '436', '437', '438', '439', '440', '441', '442', '443', '444', '445', '446', '447', '448', '449', '450', '451', '452', '453', '454', '455', '456', '457', '458', '459', '460', '461', '462', '463', '464', '465', '466', '467', '468', '469', '470', '471', '472', '473', '474', '475', '476', '477', '477A', '478', '479', '480', '481', '482', '483', '484', '485', '486', '487', '488', '489', '489A', '489B', '489C', '489D', '489E', '490', '491', '492', '493', '494', '495', '496', '497', '498', '498A', '499', '500', '501', '502', '503', '504', '505', '506', '507', '508', '509', '510', '511'
]

CrPC_list = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '25A', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '50A', '51', '52', '53', '53A', '54', '54A', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '105A', '105B', '105C', '105D', '105E', '105F', '105G', '105H', '105I', '105J', '105K', '105L', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '144A', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '164A', '165', '166', '166A', '166B', '167', '168', '169', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '198A', '198B', '199', '200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221', '222', '223', '224', '225', '226', '227', '228', '229', '230', '231', '232', '233', '234', '235', '236', '237', '238', '239', '240', '241', '242', '243', '244', '245', '246', '247', '248', '249', '250', '251', '252', '253', '254', '255', '256', '257', '258', '259', '260', '261', '262', '263', '264', '265', '265A', '265B', '265C', '265D', '265E', '265F', '265G', '265H', '265I', '265J', '265K', '265L', '266', '267', '268', '269', '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280', '281', '282', '283', '284', '285', '286', '287', '288', '289', '290', '291', '291A', '292', '293', '294', '295', '296', '297', '298', '299', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '311A', '312', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '332', '333', '334', '335', '336', '337', '338', '339', '340', '341', '342', '343', '344', '346', '347', '348', '349', '350', '351', '352', '353', '354', '355', '356', '357', '357A', '357B', '357C', '358', '359', '360', '361', '362', '363', '364', '365', '366', '367', '368', '369', '370', '371', '372', '373', '374', '375', '376', '377', '378', '379', '380', '381', '382', '383', '384', '385', '386', '387', '388', '389', '390', '391', '392', '393', '394', '396', '397', '398', '399', '400', '401', '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '412', '413', '414', '415', '416', '418', '419', '420', '421', '422', '423', '424', '425', '426', '427', '428', '429', '430', '431', '432', '433', '433A', '434', '435', '436', '436A', '437', '438', '439', '440', '441', '441A', '442', '443', '444', '445', '446', '446A', '447', '448', '449', '450', '451', '452', '453', '454', '455', '456', '457', '458', '460', '461', '462', '463', '464', '465', '466', '467', '468', '469', '470', '471', '472', '473', '474', '475', '476', '477', '478', '479', '480', '481', '482', '483', '484'
]

In [89]:
sample = [
    {
        "CNR": "CGHC010183522016",
        "case": "Applicant applied for Anticipatory-Bail.\nIs it a withdrawal application? No.\nAge of the accused is 56, 51, 32 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 438 CrPC, Section 420 IPC, Section 467 IPC, Section 468 IPC, Section 471 IPC, Section 34 IPC, Section 138 Negotiable Instruments Act, Section 139 Negotiable Instruments Act].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are As per the prosecution case a report was lodged by partner of the M/s Natural Estate that they received an information that on 20/07/2015 two cheques bearing number 605594 and 605595 of Rs.1,90,00,000/- and Rs.1,80,00,000/- have been dishonoured and on receipt of the information that cheques have been dishonoured, they enquired it from the bank. On enquiry, it was found that said cheques which were of the year 2006-2007 was given to one D.P. Gandhi he was the then partner of the Natural Estate in order to make refund to the purchasers as the company was engaged in sale and development of plots and it is alleged that the applicants who were working as Estate agents to sell plots and in connivance with such D.P. Gandhi has misused the said cheques and have interpolated the same and got it bounced. The report was made on 29/08/2015.\nArguments supporting the bail application are Learned counsel for the applicants submits that the applicants were working as Estate agents on behalf of Natural Estate to sell plots and in lieu thereof certain commission was fixed. Ultimately, cheques were given as full and final settlement to the applicants in the year 2015 which on deposit were bounced and Rs.1,90,00,000/- and Rs.1,80,00,000/- cheques have been bounced. As such, the applicants served the complainant with the notice on 27/08/2015 which was replied on 11/09/2015. Thereafter, complaint under Section 138 of Negotiable Instruments Act was preferred by the applicants against this complainant which is pending before Judicial Magistrate, Jagdalpur. He therefore submits that under the circumstances in order to avoid such liability complainant have raised this false allegation and defence may be taken before the court below. Learned counsel therefore submits that the applicants may be granted benefit of anticipatory bail.\nArguments opposing the bail application are Learned State counsel opposes the prayer for grant of bail and would submit that M/s Natural Estate initially was containing four partners Prem Raj Jain, Goutam Chand Jain, Harsh Jain and D.P. Gandhi. It is contended that D.P. Gandhi retired in the year 2012 for which necessary publication was also made and the main cheque contains signature of D.P. Gandhi along with Harsh Jain and therefore it would amount to apparent fraud committed by the applicants. He submits that all the due payment were made to the applicants and therefore by forged signature the cheque has been interpolated and the expert report confirms the fact that the letter accompanying the cheque was forged.",
        "outcome": "The outcome of the case is Bail granted. The bail conditions are [that the applicants shall make themselves available for interrogation before the investigating officer as and when required, that the applicants shall not, directly or indirectly, make any inducement, threat or promise to any person acquainted with the facts of the case so as to dissuade him/her from disclosing such facts to the Court or to any police officer, that the applicants shall not act, in any manner, which will be prejudicial to fair and expeditious trial, and the applicants shall appear before the trial Court on each and every date given to them by the said Court till disposal of the trial].",
        "reasoning": "I have perused the case diary and the voluminous documents. It is not disputed that cheque bears signature of Harsh Jain, therefore necessarily when the complaint under Section 138 of Negotiable Instruments Act is preferred at the threshold it cannot be stated that said cheques were forged as it would amount to adjudicating the complaint at the stage of bail. It is open to the complainant to raise such defence before the competent court wherein complaint has been filed. Prima facie since signature over the cheque has not been disputed that are the complainant in this case of Harsh Jain, the presumption under Section 139 of Negotiable Instruments Act would follow. Taking into documents and the dispute projected, I am inclined to extend benefit of anticipatory bail to the applicants.",
        "date_of_arrest": "not provided",
        "date_of_judgement": "22-03-2016"
    },
    {
        "CNR": "CGHC010244732020",
        "case": "Applicant applied for Anticipatory-Bail. Is it a withdrawal application? No. Age of the accused is 25, 34 years. Health issues for the accused are None. There are no past criminal records of the accused. Statutes mentioned in the judgement are [Section 304 read with 34 of IPC, Section 79 of Juvenile Justice Act, 2015, Section 304-A of IPC]. Precedents mentioned in the judgement are None. Details of the incident are Case of the prosecution is that due to failure of electrical maintenance, live electric wire caused running of electricity in the crane and the victim came in touch with the same and died at the spot. Arguments supporting the bail application are Learned counsel for the applicant would argue that the applicant G.Rama Rao, the owner of the factory and as far as maintenance part is concerned, it has to be done by the concerned Incharge of the factory. Therefore, he cannot be involved in the case on any count whatsoever though he may have other civil liability of compensation being the owner of the factory. In addition, there is no specific allegation of commission of any offence under the Factory Act. As far as Ganesh Ram Sahu, Supervisor is concerned, it is submitted that it was a case of simple accident. The worker who died at the spot was not involved in any electric work but he was engaged in the work of manufacture of cement pole but due to accident he came in touch with the crane which got in touch with a live electric wire and due to electric shock accident took place and therefore, at the most, even if it is held that Ganesh Ram Sahu did not take any steps to ensure proper electrical maintenance, the criminal overt act would not travel beyond the scope of 304-A of IPC. Arguments opposing the bail application are Learned counsel for the State and Objector oppose the prayer for grant of bail and submits that during investigation and collection of material, it has come on record in the form of statement that the workers were continuously giving information to the applicants regarding there being risk of electric shock due to improper maintenance but no care was taken. Only a day before the date of incident which was informed that even then no care was taken and running of electricity in the crane was result of the failure of maintenance and supply on the part of the applicants.",
        "outcome": "The outcome of the case is Bail granted. The bail conditions are [that the applicant-G. Rama Rao shall make himself available for interrogation by a Police Officer as and when required; that he shall not directly or indirectly make any inducement, threat or promise to any person acquainted with the facts of the case so as to dissuade him from disclosing such facts to the Court or to any Police Officer; that he shall not act, in any manner, which will be prejudicial to fair and expeditious trial; and that he shall appear before the trial Court on each and every date given to him by the said Court till disposal of the trial].",
        "reasoning": "Having considered the submission of learned counsel for the parties, as far as applicant- G. Rama Rao is concerned, he was the owner of the factory and that the report dated 20.08.2020 of Deputy Director Industrial Health and safety has stated regarding lack in maintenance as the main cause of the accident, bail application of G. Rama Rao is allowed. However, Ganesh Ram Sahu is concerned, this Court finds that even if day before the incident specific complaint was made regarding there being problem in the electrical system and one person already suffered electric shock but no steps was allegedly taken resulting in death on the next day due to electric shock, therefore, application of Ganesh Ram Sahu is rejected.",
        "date_of_arrest": "not provided",
        "date_of_judgement": "26-02-2021"
    },
    {
        "CNR": "CGHC010176152020",
        "case": "Applicant applied for Anticipatory-Bail. Is it a withdrawal application? No. Age of the accused is 50, 58, 52 years. Health issues for the accused are None. There are no past criminal records of the accused. Statutes mentioned in the judgement are [Section 498-A IPC, Section 34 IPC, Section 354 IPC, Section 506 IPC, Section 5 ChhattisgarhTonahi Pratadna Nivaran Adhiniyam, 2005, Section 6 ChhattisgarhTonahi Pratadna Nivaran Adhiniyam, 2005, Section 438 CrPC]. Precedents mentioned in the judgement are None. Details of the incident are The applicants are the father-in-law and mother-in-law of the complainant. Marriage between the complainant and Sanjay Singh i.e. son of applicants was solemnized on 17.01.2019. Allegedly, after the marriage, complainant was subjected to cruelty for demand of dowry by her father-in-law, mother-in-law, her husband and her brother-in-law. It is further alleged that applicant used to do sorcery on complainant. He also used to call her tonhi and wanted to expelled her from the house. On the basis of the report made by the complainant on 8.7.2020, offence has been registered. Arguments supporting the bail application are Learned Counsel appearing for the applicants submits that the applicants are innocent and have been falsely implicated in the present case. He further submits that applicants are the father-in-law and mother-in-law of the complainant and only general allegations have been levelled against them. Main allegations are against the husband and brother-in-law of the complainant and they have already been granted regular bail. He further submits that applicant namely Pramod Tiwari reside in the house of the complainant on rent and he used to perform puja in their house. He is falsely implicated in the present case. It is further submitted that complainant is residing separate since 19.6.2019. On 12.2.2020 husband of the complainant filed petition under Section 13 of Hindu Marriage Act and thereafter, in counter-blast, complainant lodged the report on 8.7.2020. Arguments opposing the bail application are Learned Counsel appearing for the State opposes the bail application.",
        "outcome": "The outcome of the case is Bail granted. The bail conditions are [Each applicant shall furnish a personal bond in the sum of Rs. 20,000/- with one solvent surety for the like sum to the satisfaction of the Arresting Officer/Presiding Officer of the concerned trial Court, They shall not directly or indirectly make any inducement, threat or promise to any person acquainted with the facts of the case so as to dissuade them from disclosing such fact to the Court, They shall not act in any manner which will be prejudicial to fair and expeditious trial, They shall appear before the trial Court on each and every date given to them by the said Court till disposal of the trial].",
        "reasoning": "Taking into consideration the submissions put-forth on behalf of the parties, considering the facts and circumstances of the case, evidence collected by the prosecution and particularly considering the fact that main allegations are against the husband and brother-in-law of the complainant and they have already been granted regular bail, also complainant is residing separate since 19.6.2019 and report has been lodged after filing of the divorce petition by the husband.",
        "date_of_arrest": "not provided",
        "date_of_judgement": "16-09-2020"
    },
    {
        "CNR": "CGHC010221612019",
        "case": "Applicant applied for Anticipatory-Bail.\nIs it a withdrawal application? No.\nAge of the accused is About 30 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 438 of the Cr.P.C., Section 341 of the IPC, Section 376 of the IPC, Section 506 of the IPC, Section 6 of the POCSO Act, 2012, Sections 3(2)(V), 3(2V-A) SC/ST (Prevention of Atrocities) Act].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are On 10.06.2019, a complaint was filed by the prosecutrix alleging that on 07.06.2019 at about 3 PM, the applicant, who is a teacher, asked her over telephone to visit his home at 6 PM for filling the form of Class 10th Distance Education. As per the instruction of her teacher, she visited the applicant's house and there she stayed till 9 PM. Allegedly, during that time, the applicant threatened her and committed forcible sexual intercourse with her. On the basis of said report, offence has been registered against the applicant.\nArguments supporting the bail application are Learned counsel appearing on behalf of the applicant submits that the applicant is innocent and has been falsely implicated in the present case. He further submits that the applicant is a school teacher, on 07.06.2019 complainant came to his place along with her parents and on their request applicant has filled the examination form of class 10th Distance Education and thereafter he asked her parents for depositing the tuition fee and examination fee, thereafter parents of the complainant started quarrel with the applicant and beaten him, due to which, the applicant sustained injuries and was treated at CHC Hospital. Thereafter, on 10.06.2019, after three days of incident to save themselves, they made a report against the applicant. Learned Counsel for the applicant further submits that offence under Section 6 of the POCSO is not made out because as per school record i.e. Dakhil Kharij of the prosecutrix, her date of birth is mentioned as 19.03.2001, therefore, on the date of incident i.e. 07.06.2019, she was aged bout 18 years 3 months. He further submits that during the pendency of this present application, police has also registered the offence under Sections 3(2)(V), 3(2V-A) SC/ST (Prevention of Atrocities) Act. But, there is nothing on record on the basis of which it can be said that the offence under Sections 3(2)(V), 3(2V-A) SC/ST (Prevention of Atrocities) Act can be made out against the applicant. He lastly submits that if the entire case taken as it is, it seems that the prosecutrix is a major and she was consenting party of the alleged Act because as stated by herself, she called the applicant and went his house on her own will and there she stayed till 9 PM, therefore, prima facie no case under Section 376 of the IPC can be made out against the applicant. There was a delay of three days in lodging the FIR and no proper explanation has been given by the prosecution in this regard. The applicant is a reputed person of his society, he is the permanent resident of aforementioned address and there is no chance of his absconding, therefore, he may be granted benefit of anticipatory bail.\nArguments opposing the bail application are Learned counsel appearing on behalf of State opposes the bail application.",
        "outcome": "The outcome of the case is Bail granted. The bail conditions are [That the accused/applicant shall make himself available for interrogation before the concerned Investigating Officer as and when required; The accused/applicant shall not, directly or indirectly, make any inducement, threat or promise to any person acquainted with the facts of the case so as to dissuade him/her from disclosing such facts to the Court or to any police officer; The accused/applicant shall not act, in any manner which will be prejudicial to fair and expeditious trial; The applicant shall appear before the Trial Court on each and every date given to him by the said Court till disposal of the trial].",
        "reasoning": "Considering the facts and circumstances of the case, evidence collected by the prosecution, arguments advanced by both the counsel appearing for the parties. Without further commenting on merits of the case, in my considered opinion, it is a fit case for grant of anticipatory bail to the applicant.",
        "date_of_arrest": "not provided",
        "date_of_judgement": "22-10-2019"
    },
]

In [90]:
def section_correction(section):
    # Remove underscore and everything after it if underscore is between digits or followed by digit
    if re.search(r'\d+_\d+', section) or re.search(r'_\d+', section):
        corrected_section = re.sub(r'_\d+.*', '', section)
    else:
        corrected_section = section
    # Convert letters to uppercase and remove all underscores
    corrected_section = re.sub(r'_', '', corrected_section)
    corrected_section = re.sub(r'([a-zA-Z]+)', lambda m: m.group(1).upper(), corrected_section)
    
    return corrected_section


In [119]:
def get_ipc_context(section, IPC_list, IPC_dict):
    # Retrieve context for IPC sections
    section = section_correction(section)
    if section in IPC_list:
        section_data = IPC_dict.get(section, {})
        title = section_data.get('title', '')
        description = section_data.get('description', '')
        return f"Section {section} IPC: {title}: {description}".strip()
    else:
        return None

def get_crpc_context(section, CrPC_list, CrPC_dict):
    # Retrieve context for CrPC sections
    section = section_correction(section)
    if section in CrPC_list:
        section_data = CrPC_dict.get(section, {})
        title = section_data.get('title', '')
        description = section_data.get('description', '')
        return f"Section {section} CrPC: {title}: {description}".strip()
    else:
        return None

In [120]:
get_ipc_context("304_a", IPC_list, IPC_dict)

'Section 304A IPC: Causing death by negligence: Whoever causes the death of any person by doing any rash or negligent act not amounting to culpable homicide shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.'

In [121]:
abbr = ['ipc', 'i.p.c.', 'i.p.c', 'indian penal code', 'crpc', 'cr.p.c.', 'cr.pc.', 'code of criminal procedure', 'criminal procedure code', 'I.T. Act']
long_text = "CrPC"

matches = process.extractOne(
    long_text.lower(),
    abbr,
    scorer=fuzz.partial_ratio,
    score_cutoff=70  # Adjustable!
)
print(matches)


('crpc', 100.0, 4)


In [122]:
augmented_data = []

extractor = StatuteExtractor(common_abbreviations=common_abbreviations)

abbr = ['ipc', 'i.p.c.', 'i.p.c', 'indian penal code', 'crpc', 'cr.p.c.', 'cr.pc.', 'code of criminal procedure', 'criminal procedure code', 'I.T. Act']

for item in tqdm(kachha_data, desc="adding statutes context"):
    # print("Processing item:", item.get('CNR', 'Unknown CNR'))
    text = item.get('case', '')
    statutes = extractor.extract(text)
    new_item = item.copy()
    new_item['context'] = []
    new_item['statutes'] = []
    # print("Extracted Statutes:", statutes)   

    for act in statutes:
        act_name_query = act['act']
        section = act['section']
        # print("Section:", section, "Act Name Query:", act_name_query)

        ## if act is CrPC or IPC
        match = process.extractOne(
            act_name_query.lower(),
            abbr,
            scorer=fuzz.partial_ratio,
            score_cutoff=70  # Adjustable!
        )
        # print("IPC/CrPC Match:", match)
        if match:
            act_name_query = match[0]
            if act_name_query.lower() in ['ipc', 'i.p.c.', 'i.p.c', 'indian penal code', ]:
                new_item['statutes'].append(section_correction(section)+ ' IPC')
                context = get_ipc_context(section, IPC_list, IPC_dict)
                if context: new_item['context'].append(context)
                # print(new_item['context'])
                continue
                
            elif act_name_query.lower() in ['crpc', 'cr.p.c.','cr.pc.', 'code of criminal procedure', 'criminal procedure code']:
                new_item['statutes'].append(section_correction(section)+ ' CrPC')
                context = get_crpc_context(section, CrPC_list, CrPC_dict)
                if context: new_item['context'].append(context)
                # print(new_item['context'])
                continue
                
        ## else search for canonical act name acts list
        # -- first check for abbreviations
        matches = process.extractOne(
            act_name_query.lower(),
            [name.lower() for name in act_names],
            scorer=fuzz.partial_ratio,
            score_cutoff=70  # Adjustable!
        )
        act_name = None
        if matches:
            act_name = matches[0]
        else:
            possible_acts = match_statute_name(
                section+" "+act_name_query, 
                act_embeddings, 
                act_names, 
                return_top_k=1, 
                score_threshold=0.7
            )
            # print("Possible Acts:", possible_acts)
            if possible_acts:
                act_name = possible_acts[0]['act_name']
                
        ## now extract the data about the act
        if act_name:
            new_item['statutes'].append(section + ' ' + act_name)
            if len(act_name) and act_name in act_2_id:
                act_id = act_2_id[act_name]
                act_info = id_2_text.get(str(act_id), {})
                if act_info:
                    sections = act_info.get('Sections', {})
                    if section in sections:
                        new_item['context'].append(f"Section {section} {act_name}: {sections[section]}")
    augmented_data.append(new_item)
    # print("\n")


adding statutes context: 100%|██████████| 208292/208292 [04:20<00:00, 800.86it/s]  


In [123]:
augmented_data[0]

{'case': "Applicant applied for Anticipatory-Bail.\nIs it a withdrawal application? No.\nAge of the accused is 40 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 341 IPC, Section 376 IPC, Section 506 IPC, Section 511 IPC, Section 366 r/w 34 IPC].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are The accused are charged with offences related to a maid servant who was allegedly physically assaulted and raped by the accused. The maid servant was working in the house of the first accused and was allegedly tied up and assaulted by the accused while the first accused's wife was not present.\nArguments supporting the bail application are The allegations made against the first accused are baseless and false. The de facto complainant had a motive to falsely implicate the first accused in the crime. The first accused lost money from the house and had suspicions about 

In [124]:
with open("./data/final_cleaned_augmented.json", "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, ensure_ascii=False, indent=2)